# Algoritmo KMP (Knuth-Morris-Pratt)

Busca un patrón en un texto en $O(n + m)$ sin retroceder en el texto. Precalcula la tabla de fallos: para cada prefijo del patrón, la longitud del borde más largo (prefijo propio que también es sufijo). Al fallar una comparación, el patrón salta usando esa tabla.

## 1. Tabla de fallos

Calcular la tabla de bordes del patrón.

In [1]:
def tabla_fallos(patron):
    fallos, k = [0] * len(patron), 0
    for i in range(1, len(patron)):
        while k and patron[i] != patron[k]:
            k = fallos[k - 1]
        if patron[i] == patron[k]:
            k += 1
        fallos[i] = k
    return fallos

print(tabla_fallos("ABABCABAB"), tabla_fallos("AAAA"))

[0, 0, 1, 2, 0, 1, 2, 3, 4] [0, 1, 2, 3]


## 2. Búsqueda de todas las apariciones

Devolver todas las posiciones donde aparece el patrón en el texto (incluso solapadas) y comprobar contra una búsqueda ingenua.

In [2]:
import random

def kmp(texto, patron):
    fallos, k, posiciones = tabla_fallos(patron), 0, []
    for i, c in enumerate(texto):
        while k and c != patron[k]:
            k = fallos[k - 1]
        if c == patron[k]:
            k += 1
        if k == len(patron):
            posiciones.append(i - k + 1)
            # sigue buscando solapadas
            k = fallos[k - 1]
    return posiciones

def ingenua(texto, patron):
    return [i for i in range(len(texto) - len(patron) + 1) if texto[i:i + len(patron)] == patron]

for _ in range(300):
    texto = "".join(random.choices("ab", k=30))
    patron = "".join(random.choices("ab", k=random.randint(1, 4)))
    assert kmp(texto, patron) == ingenua(texto, patron)
print(kmp("ABABDABACDABABCABAB", "ABABCABAB"), kmp("AAAAA", "AA"))

[10] [0, 1, 2, 3]


## 3. Contar apariciones

Contar cuántas veces aparece un patrón en un texto, con y sin solapamiento. Sin solapamiento, al completar una aparición el estado vuelve a 0 en lugar de seguir con el borde. Comprobar con la búsqueda ingenua y con `str.count`, que no cuenta las solapadas.

In [3]:
def cuenta_kmp(texto, patron, solapadas=True):
    fallos, k, total = tabla_fallos(patron), 0, 0
    for c in texto:
        while k and c != patron[k]:
            k = fallos[k - 1]
        if c == patron[k]:
            k += 1
        if k == len(patron):
            total += 1
            k = fallos[k - 1] if solapadas else 0
    return total

random.seed(6)
for _ in range(500):
    texto = "".join(random.choices("ab", k=30))
    patron = "".join(random.choices("ab", k=random.randint(1, 4)))
    assert cuenta_kmp(texto, patron) == len(ingenua(texto, patron))
    assert cuenta_kmp(texto, patron, solapadas=False) == texto.count(patron)
print(cuenta_kmp("abababab", "abab"), cuenta_kmp("abababab", "abab", solapadas=False))

3 2


## 4. Período mínimo de una cadena

El período mínimo de $s$ es el menor $p$ tal que $s_i = s_{i+p}$ para todo $i$, y vale $p = n - \text{fallos}[n-1]$: el borde más largo es lo que se solapa al desplazar la cadena $p$ posiciones. Si además $p$ divide a $n$, la cadena es una repetición de su prefijo de longitud $p$. Comprobarlo con fuerza bruta.

In [4]:
def periodo(cadena):
    return len(cadena) - tabla_fallos(cadena)[-1]

def periodo_bruto(cadena):
    n = len(cadena)
    return next(p for p in range(1, n + 1) if all(cadena[i] == cadena[i + p] for i in range(n - p)))

for _ in range(500):
    cadena = "".join(random.choices("ab", k=random.randint(1, 12)))
    assert periodo(cadena) == periodo_bruto(cadena)

for cadena in ("abcabcabc", "abcabcab", "aaaa", "abcd"):
    p = periodo(cadena)
    repeticion = f"= {cadena[:p]!r} × {len(cadena) // p}" if len(cadena) % p == 0 and p < len(cadena) else ""
    print(f"{cadena:10} período {p} {repeticion}")

abcabcabc  período 3 = 'abc' × 3
abcabcab   período 3 
aaaa       período 1 = 'a' × 4
abcd       período 4 


## 5. Rotaciones de una cadena

Una cadena $b$ es una rotación de $a$ si tienen la misma longitud y $b$ aparece dentro de $a + a$. Decidirlo con KMP en $O(n)$ y comprobarlo generando todas las rotaciones.

In [5]:
def es_rotacion(a, b):
    return len(a) == len(b) and len(kmp(a + a, b)) > 0

for _ in range(500):
    a = "".join(random.choices("ab", k=random.randint(1, 8)))
    b = "".join(random.choices("ab", k=len(a)))
    assert es_rotacion(a, b) == (b in {a[i:] + a[:i] for i in range(len(a))})
print(es_rotacion("abcde", "cdeab"), es_rotacion("abcde", "abced"))

True False
